In [ ]:
# ! pip install -q roboflow ultralytics supervision ipywidgets 

In [ ]:
from roboflow import Roboflow
from ultralytics import YOLO
from IPython.display import Image
import cv2
import datetime
import supervision as sv 

In [ ]:
# Download the dataset

rf = Roboflow(api_key="your-api-key")
project = rf.workspace("your-workspace-name").project("your-project-name")
version = project.version(2)  # replace with your version number
dataset = version.download("yolo26")


In [ ]:
# Verify the data.yaml file, edit the file if needed.

In [ ]:
# Basic training
# Requires a T4 GPU
model = YOLO ("yolo26s.pt")

In [ ]:
dataset.location 

In [ ]:
results = model.train (data = f"{dataset.location}/data.yaml", epochs = 100) 

In [ ]:
# Download the best.pt weights to your laptop

In [ ]:
# Advanced training (Colab Pro is required)
# Requires an A100 GPU
model = YOLO ("yolo26x.pt")
results = model.train (data = f"{dataset.location}/data.yaml", epochs = 100)

In [ ]:
# Download the best.pt weights to your laptop

In [ ]:
# Display the results
Image (filename=f"/content/runs/detect/train/confusion_matrix.png", width=600)

In [ ]:
Image (filename=f"/content/runs/detect/train/results.png", width=600)

In [ ]:
# Test the trained model

In [ ]:
# Upload the input video

In [ ]:
# Load the best weights model.
model = YOLO ("/path-to-your-best-weights/best.pt") 

In [ ]:
# Setup annotation utilities
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator() 

In [ ]:
# Setup input and output videos.
input_video_path  = "input-video.mp4"
output_video_path = "output-video.mp4"

cap = cv2.VideoCapture (input_video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

print(f"total_frames = {total_frames}")
print(f"frame width = {frame_width}")
print(f"frame height = {frame_height}")
print(f"fps = {fps}")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# fps = 1  # Slow down the output video for easier analysis, if needed.
writer = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))


In [ ]:
# Loop over the video frames
counter = 0 # frame counter

while cap.isOpened():
    start = datetime.datetime.now()
    ret, frame = cap.read()
    if not ret: break
    counter += 1

    results = model (frame, verbose=False)[0]

    # Annotate a copy of the original frame
    annotated_frame = frame.copy() 

    detections = sv.Detections.from_ultralytics(results)
    # Apply NMS if needed (NOT needed by YOLO26 models as NMS is applied by default)
    # detections = detections.with_nms() 
    annotated_frame = box_annotator.annotate(scene=annotated_frame, detections=detections)
    annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections)
    writer.write(annotated_frame)
    end = datetime.datetime.now()
    # print(f"Time to process frame {counter}: {(end - start).total_seconds() * 1000:.0f} miliseconds")

cap.release()
writer.release() 
